In [ ]:
%%capture
!pip install -U bitsandbytes>=0.46.1 accelerate transformers

# **1. Imports**

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import cv2
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
import seaborn as sns
import pydicom
import torch
import json
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import os

# **2. EDA**

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')

split_target = df['MCL'].fillna(-1)

In [ ]:
df.head()

## **2.1 Target Distribution & Label Imbalance**  

In [ ]:
targets = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', 'Baker\'s', 'Contusion', 'Fracture']

# Filter to only the 58 ground-truth labeled rows (where targets are not null)
ground_truth_df = df.dropna(subset=targets)

# Calculate prevalence (mean of 1s and 0s)
prevalence = ground_truth_df[targets].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
import seaborn as sns

# Updated barplot code
sns.barplot(
    x=prevalence.values, 
    y=prevalence.index, 
    hue=prevalence.index, 
    palette="viridis", 
    legend=False
)

plt.title('Prevalence of Abnormalities in Ground Truth Data (58 Studies)')
plt.xlabel('Percentage Positive')
plt.xlim(0, 1.0)
plt.show()

## **2.2 Report Duplication & Leak Risk**  

In [ ]:
plt.figure(figsize=(10, 5))
# Get the top 10 most duplicated exact report strings
report_counts = df['Report'].value_counts().head(10)

sns.barplot(x=report_counts.values, y=report_counts.index, hue=report_counts.index, palette="rocket", legend=False)
plt.title('Top 10 Most Duplicated Radiology Reports (Data Leak Risk)')
plt.xlabel('Number of Studies with Identical Text')
plt.yticks([])
plt.show()

# **3. Validation Strategy & Split Setup**

In [ ]:

df['report_hash'] = df['Report'].apply(lambda x: hashlib.md5(str(x).encode()).hexdigest())

# 4. Use StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

df['fold'] = -1

# FIX: Pass only the StudyInstanceUID column (which has no NaNs) as X
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X=df['StudyInstanceUID'], y=split_target, groups=df['report_hash'])):
    df.loc[val_idx, 'fold'] = fold

# Check that the actual labeled positive cases are balanced across folds
print("Positive MCL ratio per fold (labeled data only):")
print(df[df['MCL'].notna()].groupby('fold')['MCL'].mean())

# **4. Metadata Preprocessing**

In [ ]:
series_df = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv")
series_df = series_df.drop(columns=['Fat_Suppression']) # It's redundant

In [ ]:
plt.figure(figsize=(10, 5))
# Plot the frequency of each physical MRI orientation
sns.countplot(
    data=series_df, 
    y='Anatomical_Plane', 
    order=series_df['Anatomical_Plane'].value_counts().index, 
    palette='rocket',
    hue='Anatomical_Plane',
    legend=False
)
plt.title('Distribution of MRI Planes (Anatomical_Plane)')
plt.xlabel('Number of Scans')
plt.ylabel('Imaging Plane')
plt.show()

# **5. Computer Vision Preprocessing & Tissue Isolation**

## **5.1 Auto-Cropping Function**  

In [ ]:
def crop_to_tissue(image_array, margin=10):
    """
    Takes a 2D numpy array (MRI slice), finds the actual tissue, 
    and crops out the black background.
    """
    # Create a binary mask of pixels that are not black/background noise
    # (Threshold might need tuning based on DICOM windowing)
    binary_mask = image_array > np.percentile(image_array, 5) 
    
    # Find the bounding box of the tissue
    coords = np.argwhere(binary_mask)
    if coords.size == 0:
        return image_array # Return original if completely empty
        
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    
    # Add a small safety margin
    y_min = max(0, y_min - margin)
    y_max = min(image_array.shape[0], y_max + margin)
    x_min = max(0, x_min - margin)
    x_max = min(image_array.shape[1], x_max + margin)
    
    # Crop the image
    cropped_img = image_array[y_min:y_max, x_min:x_max]
    
    # Resize back to your model's expected input size (e.g., 224x224 or 336x336)
    resized_img = cv2.resize(cropped_img, (336, 336))
    
    return resized_img

## **5.2 DICOM Inspection: Raw vs. Cropped ROI**  

In [ ]:
sample_dcm_path = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series/1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585/1.2.826.0.1.3680043.8.498.27845969797953115400959653384511165237/1.2.826.0.1.3680043.8.498.12367777279000914615462751196361349524.dcm'

# Load DICOM and extract pixel array
dicom = pydicom.dcmread(sample_dcm_path)
pixel_array = dicom.pixel_array

# Normalize to 0-255 for visualization
img_normalized = cv2.normalize(pixel_array, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

# Apply the crop function
img_cropped = crop_to_tissue(img_normalized)

# Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img_normalized, cmap='bone')
axes[0].set_title(f'Raw DICOM (Shape: {img_normalized.shape})')
axes[0].axis('off')

axes[1].imshow(img_cropped, cmap='bone')
axes[1].set_title(f'Cropped & Resized (Shape: {img_cropped.shape})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## **5.3 Target Co-Occurrence Matrix**  

In [ ]:
plt.figure(figsize=(10, 8))
# Calculate correlation matrix strictly on the 58 expert-labeled studies
corr_matrix = ground_truth_df[targets].corr()

# Plot the heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Co-Occurrence Correlation of Knee Abnormalities")
plt.show()

## **5.4 3D Volume Sequence Visualization** 

In [ ]:
def plot_slice_sequence(series_dir, num_slices=5):
    # Fetch all DICOMs in the series
    dicom_paths = [os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.endswith('.dcm')]
    
    # Sort by the actual InstanceNumber stored inside the DICOM metadata
    def get_instance_number(path):
        dcm = pydicom.dcmread(path, stop_before_pixels=True)
        return int(dcm.InstanceNumber)
        
    sorted_paths = sorted(dicom_paths, key=get_instance_number)
    
    # Select evenly spaced slices across the volume
    step = max(1, len(sorted_paths) // num_slices)
    selected_paths = [sorted_paths[i * step] for i in range(num_slices)]
    
    fig, axes = plt.subplots(1, num_slices, figsize=(18, 4))
    for i, path in enumerate(selected_paths):
        img = pydicom.dcmread(path).pixel_array
        
        # FIX: Use OpenCV's native CV_8U constant instead of NumPy's uint8
        img_norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        img_cropped = crop_to_tissue(img_norm)
        
        axes[i].imshow(img_cropped, cmap='bone')
        axes[i].set_title(f"Slice {get_instance_number(path)}")
        axes[i].axis('off')
        
    plt.suptitle("Sequential Progression of MRI Slices")
    plt.tight_layout()
    plt.show()

# Pass the directory of your sample_dcm_path
series_folder = os.path.dirname(sample_dcm_path)
plot_slice_sequence(series_folder)

##  **5.5 Z-Axis Depth (Slice Count per Series)**

In [ ]:
base_dir = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series'

sample_studies = os.listdir(base_dir)[:1000]
slice_counts = []

for study in tqdm(sample_studies, desc="Calculating Z-Axis Depths"):
    study_path = os.path.join(base_dir, study)
    for series in os.listdir(study_path):
        series_path = os.path.join(study_path, series)
        if os.path.isdir(series_path):
            num_slices = len([f for f in os.listdir(series_path) if f.endswith('.dcm')])
            slice_counts.append(num_slices)

plt.figure(figsize=(10, 5))
sns.histplot(slice_counts, bins=30, kde=True, color='orange')
plt.title('Distribution of Slices per MRI Series (Z-Axis Depth)')
plt.xlabel('Number of Slices in Volume')
plt.ylabel('Frequency')
plt.show()

## **5.6 Radiology Report Length Distribution**

In [ ]:
# Calculate word count for each report
df['word_count'] = df['Report'].fillna("").apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(df['word_count'], bins=50, kde=True, color='purple')
plt.title('Distribution of Radiology Report Lengths (Word Count)')
plt.xlabel('Number of Words per Report')
plt.ylabel('Frequency')
plt.show()